## Teste com variáveis selecionadas - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
# Garante que não perderemos dados devido a falhas do pluviômetro e radiação noturna
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)

# Interpolação bidirecional para costurar os picotes da Umidade
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Cinturão de segurança para o apagão inicial do sensor de vento em Janeiro
colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

# Poda final caso o próprio termômetro (target) tenha falhado
df_model = df_model.dropna()

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("=====================================================================")
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C1 ---")
print("=====================================================================")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n=====================================================================")
print("--- RESULTADOS RANDOM FOREST - BERTIOGA C1 ---")
print("=====================================================================")
rf = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")
print(f"MAE do Random Forest: {mae_rf:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf:.4f} °C")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C1 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.689
Model:                                                      OLS   Adj. R-squared:                  0.689
Method:                                           Least Squares   F-statistic:                     2588.
Date:                                          Wed, 01 Jul 2026   Prob (F-statistic):               0.00
Time:                                                  22:50:21   Log-Likelihood:                -15717.
No. Observations:                                          7008   AIC:                         3.145e+04
Df Residuals:                                              7001   BIC:                         3.150e+04
Df Model:                                                     6                                         
Cov

## Teste com todas as variáveis (Sem filtro) - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# CENÁRIO 2: 11 Variáveis Físicas (Sem Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("=====================================================================")
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C2 ---")
print("=====================================================================")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n=====================================================================")
print("--- RESULTADOS RANDOM FOREST - BERTIOGA C2 ---")
print("=====================================================================")
rf = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")
print(f"MAE do Random Forest: {mae_rf:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf:.4f} °C")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C2 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.700
Model:                                                      OLS   Adj. R-squared:                  0.699
Method:                                           Least Squares   F-statistic:                     1481.
Date:                                          Wed, 01 Jul 2026   Prob (F-statistic):               0.00
Time:                                                  23:01:58   Log-Likelihood:                -15599.
No. Observations:                                          7008   AIC:                         3.122e+04
Df Residuals:                                              6996   BIC:                         3.130e+04
Df Model:                                                    11                                         
Cov

## Teste com variáveis selecionadas + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# CENÁRIO 3: 6 Variáveis Físicas Limpas + Hora e Mês
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("=====================================================================")
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C3 ---")
print("=====================================================================")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n=====================================================================")
print("--- RESULTADOS RANDOM FOREST - BERTIOGA C3 ---")
print("=====================================================================")
rf = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")
print(f"MAE do Random Forest: {mae_rf:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf:.4f} °C")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)
print("=====================================================================")

--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C3 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.755
Model:                                                      OLS   Adj. R-squared:                  0.755
Method:                                           Least Squares   F-statistic:                     2696.
Date:                                          Wed, 01 Jul 2026   Prob (F-statistic):               0.00
Time:                                                  23:14:04   Log-Likelihood:                -14884.
No. Observations:                                          7008   AIC:                         2.979e+04
Df Residuals:                                              6999   BIC:                         2.985e+04
Df Model:                                                     8                                         
Cov

## Teste com todas as variáveis (Sem filtro) + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# CENÁRIO 4: Todas as 11 Variáveis Físicas + Hora e Mês
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("=====================================================================")
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C4 ---")
print("=====================================================================")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n=====================================================================")
print("--- RESULTADOS RANDOM FOREST - BERTIOGA C4 ---")
print("=====================================================================")
rf = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")
print(f"MAE do Random Forest: {mae_rf:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf:.4f} °C")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)
print("=====================================================================")

--- RESULTADOS ESTATÍSTICOS (OLS) - BERTIOGA C4 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.758
Model:                                                      OLS   Adj. R-squared:                  0.758
Method:                                           Least Squares   F-statistic:                     1690.
Date:                                          Wed, 01 Jul 2026   Prob (F-statistic):               0.00
Time:                                                  23:31:55   Log-Likelihood:                -14834.
No. Observations:                                          7008   AIC:                         2.970e+04
Df Residuals:                                              6994   BIC:                         2.979e+04
Df Model:                                                    13                                         
Cov

## Teste com variáveis selecionadas - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança (vírgula para ponto)
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 1: Física Pura, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção contra o apagão do anemômetro no início do ano
colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 1) - BERTIOGA")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do passo para correção de erros)
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 1) - BERTIOGA

R² do Gradient Boosting em dados futuros: 0.7232

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    58.513820
RADIACAO GLOBAL (Kj/m²)                                  29.085504
UMIDADE RELATIVA DO AR, HORARIA (%)                      11.148968
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.599874
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.482268
VENTO, VELOCIDADE HORARIA (m/s)                           0.169566
dtype: float64


## Teste com todas as variáveis (Sem filtro) - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 2: 11 Variáveis Físicas, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção contra o apagão do anemômetro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 2) - BERTIOGA")
print("=======================================================\n")

gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 2) - BERTIOGA

R² do Gradient Boosting em dados futuros: 0.7326

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    41.606907
RADIACAO GLOBAL (Kj/m²)                                  28.528950
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          14.268710
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  5.769757
UMIDADE RELATIVA DO AR, HORARIA (%)                       3.728289
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.415901
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          1.493157
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.529968
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.413953
VENTO, VELOCIDADE HORARIA (m/s)                           0.181717
VENTO, RAJADA MAXIMA (m/s)                                0.062690
dtype: float64


## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança (vírgula para ponto)
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 3: 6 Variáveis Físicas Limpas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção contra o apagão do anemômetro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 3) - BERTIOGA")
print("=======================================================\n")

gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 3) - BERTIOGA

R² do Gradient Boosting em dados futuros: 0.7135

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    34.546404
Mes                                                      28.098480
RADIACAO GLOBAL (Kj/m²)                                  15.440155
UMIDADE RELATIVA DO AR, HORARIA (%)                      14.482953
Hora                                                      7.102960
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.199125
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.126745
VENTO, VELOCIDADE HORARIA (m/s)                           0.003177
dtype: float64


## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 4: Todas as 11 Físicas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção contra o apagão do anemômetro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 4) - BERTIOGA")
print("=======================================================\n")

gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 4) - BERTIOGA

R² do Gradient Boosting em dados futuros: 0.7241

Importância das Variáveis (em %):
Mes                                                      31.166571
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    28.835176
RADIACAO GLOBAL (Kj/m²)                                  13.564435
UMIDADE RELATIVA DO AR, HORARIA (%)                       8.508283
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  6.104501
Hora                                                      5.498127
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.701544
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          1.306177
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           1.019971
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.170885
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.114740
VENTO, RAJADA MAXIMA (m/s)                                0.009592
VENTO, VELOCIDADE HORARIA (m/s)                           0.00

## Teste com variáveis selecionadas - XGBoost


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança para o padrão numérico decimal
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 1: Física pura, sem tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Cinturão de Segurança - Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)

# Interpolação linear para corrigir os picotes de umidade ao longo das frentes frias
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Tratando os NaNs iniciais do anemômetro (sensor de vento) para evitar descarte de linhas
colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

# Remoção final apenas se o termômetro principal falhou
df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica Rigorosa (80% Treino / 20% Teste Futuro)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 1) - BERTIOGA")
print("=======================================================\n")

# Mantendo consistência metodológica: 100 estimadores e taxa de aprendizado de 0.1
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)
mae_xgb = mean_absolute_error(y_test, y_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")
print(f"MAE do XGBoost: {mae_xgb:.4f} °C")
print(f"RMSE do XGBoost: {rmse_xgb:.4f} °C")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em % de contribuição):")
print(importances_xgb * 100)
print("=======================================================")

MODELO: XGBOOST (CENÁRIO 1) - BERTIOGA

R² do XGBoost em dados futuros: 0.6858
MAE do XGBoost: 1.5263 °C
RMSE do XGBoost: 1.9465 °C

Importância das Variáveis (em % de contribuição):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    44.011898
RADIACAO GLOBAL (Kj/m²)                                  32.782806
UMIDADE RELATIVA DO AR, HORARIA (%)                      12.865749
VENTO, VELOCIDADE HORARIA (m/s)                           3.894193
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          3.398445
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      3.046907
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança (vírgula para ponto)
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 2: 11 Variáveis Físicas, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção contra o apagão do anemômetro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 2) - BERTIOGA")
print("=======================================================\n")

xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)
mae_xgb = mean_absolute_error(y_test, y_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")
print(f"MAE do XGBoost: {mae_xgb:.4f} °C")
print(f"RMSE do XGBoost: {rmse_xgb:.4f} °C")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em % de contribuição):")
print(importances_xgb * 100)
print("=======================================================")

MODELO: XGBOOST (CENÁRIO 2) - BERTIOGA

R² do XGBoost em dados futuros: 0.7269
MAE do XGBoost: 1.4308 °C
RMSE do XGBoost: 1.8148 °C

Importância das Variáveis (em % de contribuição):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    31.118315
RADIACAO GLOBAL (Kj/m²)                                  24.011301
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          15.742630
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  7.968639
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          5.528982
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  4.753686
UMIDADE RELATIVA DO AR, HORARIA (%)                       3.809608
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          2.385571
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      2.083635
VENTO, VELOCIDADE HORARIA (m/s)                           1.568548
VENTO, RAJADA MAXIMA (m/s)                                1.029094
dtype: float32


## Teste com variáveis selecionadas + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 3: 6 Variáveis Físicas Limpas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado (Cinturão de Segurança - Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção do sensor de vento
colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 3) - BERTIOGA")
print("=======================================================\n")

xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)
mae_xgb = mean_absolute_error(y_test, y_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")
print(f"MAE do XGBoost: {mae_xgb:.4f} °C")
print(f"RMSE do XGBoost: {rmse_xgb:.4f} °C")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em % de contribuição):")
print(importances_xgb * 100)
print("=======================================================")

MODELO: XGBOOST (CENÁRIO 3) - BERTIOGA

R² do XGBoost em dados futuros: 0.7237
MAE do XGBoost: 1.4612 °C
RMSE do XGBoost: 1.8254 °C

Importância das Variáveis (em % de contribuição):
Mes                                                      27.264553
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    26.165434
RADIACAO GLOBAL (Kj/m²)                                  24.667038
UMIDADE RELATIVA DO AR, HORARIA (%)                      12.739305
Hora                                                      6.164320
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.472114
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.977066
VENTO, VELOCIDADE HORARIA (m/s)                           0.550172
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 4: Todas as 11 Físicas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento Avançado (Cinturão de Segurança - Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

# Proteção do sensor de vento
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 4) - BERTIOGA")
print("=======================================================\n")

xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)
mae_xgb = mean_absolute_error(y_test, y_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")
print(f"MAE do XGBoost: {mae_xgb:.4f} °C")
print(f"RMSE do XGBoost: {rmse_xgb:.4f} °C")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em % de contribuição):")
print(importances_xgb * 100)
print("=======================================================")

MODELO: XGBOOST (CENÁRIO 4) - BERTIOGA

R² do XGBoost em dados futuros: 0.7438
MAE do XGBoost: 1.4029 °C
RMSE do XGBoost: 1.7579 °C

Importância das Variáveis (em % de contribuição):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    29.150253
Mes                                                      19.025118
RADIACAO GLOBAL (Kj/m²)                                  18.499323
UMIDADE RELATIVA DO AR, HORARIA (%)                       8.546246
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  6.601948
Hora                                                      5.475707
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  4.556810
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.945023
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           2.200479
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.941841
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.877235
VENTO, RAJADA MAXIMA (m/s)                                0.606000
VENTO, VELOCI

## Teste com variáveis selecionadas - XGBoost tunado


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df_tune1 = pd.read_excel(nome_arquivo)

# Conversão segura de strings decimais
for col in df_tune1.columns:
    if df_tune1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune1[col] = pd.to_numeric(df_tune1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune1[col] = pd.to_numeric(df_tune1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_tune1[[target_col] + X_cols_1].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model_1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear', limit_direction='both')

colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model_1[col_v] = df_model_1[col_v].fillna(df_model_1[col_v].median())

df_model_1 = df_model_1.dropna(subset=[target_col])

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO) - BERTIOGA")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],          # Número de árvores
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Passo de aprendizado
    'max_depth': [3, 4, 5, 6],                # Profundidade máxima de cada árvore
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # % de variáveis usadas por árvore
    'subsample': [0.7, 0.8, 0.9, 1.0]         # % de dados usados por árvore
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório (testará 100 combinações diferentes)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_1, y_train_1)
melhor_xgb_1 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_1 = melhor_xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(melhor_xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO) - BERTIOGA
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
R² do XGBoost Otimizado em dados futuros: 0.7026

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    41.441544
RADIACAO GLOBAL (Kj/m²)                                  34.341362
UMIDADE RELATIVA DO AR, HORARIA (%)                      14.632309
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      3.885551
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          3.453843
VENTO, VELOCIDADE HORARIA (m/s)                           2.245390
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df_tune2 = pd.read_excel(nome_arquivo)

# Conversão segura de strings decimais
for col in df_tune2.columns:
    if df_tune2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune2[col] = pd.to_numeric(df_tune2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune2[col] = pd.to_numeric(df_tune2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: 11 Variáveis Físicas (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_tune2[[target_col] + X_cols_2].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model_2[col_v] = df_model_2[col_v].fillna(df_model_2[col_v].median())

df_model_2 = df_model_2.dropna(subset=[target_col])

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - FÍSICA + HISTÓRICO) - BERTIOGA")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_2, y_train_2)
melhor_xgb_2 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_2 = melhor_xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(melhor_xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)
print("=======================================================")

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - FÍSICA + HISTÓRICO) - BERTIOGA
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
R² do XGBoost Otimizado em dados futuros: 0.6970

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    29.671501
RADIACAO GLOBAL (Kj/m²)                                  21.768015
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          19.552811
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  7.714976
UMIDADE RELATIVA DO AR, HORARIA (%)                       4.659336
VENTO, VELOCIDADE HORARIA (m/s)                           4.649286
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  4.087924
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          3.275339
PRECIPITAÇÃO TOTAL, HOR

## Teste com variáveis selecionadas + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df_tune3 = pd.read_excel(nome_arquivo)

# Conversão segura de strings decimais
for col in df_tune3.columns:
    if df_tune3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune3[col] = pd.to_numeric(df_tune3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune3[col] = pd.to_numeric(df_tune3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 3: 6 Variáveis Físicas Limpas + Tempo
df_tune3['Hora'] = df_tune3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune3['Mes'] = pd.to_datetime(df_tune3['Data']).dt.month

X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_tune3[[target_col] + X_cols_3].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_model_3[col_v] = df_model_3[col_v].fillna(df_model_3[col_v].median())

df_model_3 = df_model_3.dropna(subset=[target_col])

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 3. Divisão Cronológica
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - FÍSICA LIMPA + TEMPO) - BERTIOGA")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_3, y_train_3)
melhor_xgb_3 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_3 = melhor_xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(melhor_xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)
print("=======================================================")

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - FÍSICA LIMPA + TEMPO) - BERTIOGA
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
R² do XGBoost Otimizado em dados futuros: 0.6968

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    29.777018
Mes                                                      27.004633
UMIDADE RELATIVA DO AR, HORARIA (%)                      15.869713
RADIACAO GLOBAL (Kj/m²)                                  15.160736
Hora                                                      9.315846
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.149452
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.876001
VENTO, VELOCIDADE HORARIA (m/s)                           0.846597
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df_tune4 = pd.read_excel(nome_arquivo)

# Conversão segura de strings decimais
for col in df_tune4.columns:
    if df_tune4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune4[col] = pd.to_numeric(df_tune4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune4[col] = pd.to_numeric(df_tune4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 4: Todas as 11 Físicas + Tempo
df_tune4['Hora'] = df_tune4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune4['Mes'] = pd.to_datetime(df_tune4['Data']).dt.month

X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_tune4[[target_col] + X_cols_4].copy()

# Tratamento Avançado de Dados Faltantes (Data Quality Fix - Padrão Litoral)
df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model_4[col_v] = df_model_4[col_v].fillna(df_model_4[col_v].median())

df_model_4 = df_model_4.dropna(subset=[target_col])

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 3. Divisão Cronológica
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - CAOS TOTAL) - BERTIOGA")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_4, y_train_4)
melhor_xgb_4 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_4 = melhor_xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(melhor_xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)
print("=======================================================")

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - CAOS TOTAL) - BERTIOGA
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.9, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
R² do XGBoost Otimizado em dados futuros: 0.7303

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    24.888580
Mes                                                      15.478054
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 12.695439
RADIACAO GLOBAL (Kj/m²)                                  12.591391
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                 10.792289
UMIDADE RELATIVA DO AR, HORARIA (%)                       7.790637
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           6.138655
Hora                                                      4.964171
PRESSÃO ATMOSFERICA MIN. NA HOR

## Teste com variáveis selecionadas - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança para numérico
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 1
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

# Isolando e aplicando o Cinturão de Segurança (Padrão Litoral)
df_pca1 = df[X_cols_1].copy()
df_pca1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca1['RADIACAO GLOBAL (Kj/m²)'] = df_pca1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca1 = df_pca1.interpolate(method='linear', limit_direction='both')

# Protegendo as colunas de vento do apagão de janeiro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_pca1[col_v] = df_pca1[col_v].fillna(df_pca1[col_v].median())

df_pca1 = df_pca1.dropna()

# 3. Padronização Obrigatória (Z-score)
# O PCA é extremamente sensível à escala, então precisamos colocar pressão (1012) e chuva (0.0) na mesma proporção.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca1)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BERTIOGA - CENÁRIO 1 (SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_1))],
    index=X_cols_1
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1 (o eixo mais importante)
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BERTIOGA - CENÁRIO 1 (SEM TEMPO)

Variância no Componente 1 (PC1): 29.85%
Variância no Componente 2 (PC2): 22.34%
Variância Acumulada (PC1 + PC2): 52.19%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.564902   
RADIACAO GLOBAL (Kj/m²)                            -0.564723   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.481974   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.356666   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.049291   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.005821   

                                                    Impacto_Absoluto_PC1  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.564902  
RADIACAO GLOBAL (Kj/m²)                                         0.564723  
VENTO, VELOCIDADE HORARIA (m/s)                                 0.481974  
VENTO, DIREÇÃO HORARIA (gr) (

## Teste com todas as variáveis (Sem filtro) - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança para numérico
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 2 (11 Variáveis Físicas)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

# Isolando e aplicando o Cinturão de Segurança (Padrão Litoral)
df_pca2 = df[X_cols_2].copy()
df_pca2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca2['RADIACAO GLOBAL (Kj/m²)'] = df_pca2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca2 = df_pca2.interpolate(method='linear', limit_direction='both')

# Protegendo as colunas de vento do apagão de janeiro
colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_pca2[col_v] = df_pca2[col_v].fillna(df_pca2[col_v].median())

df_pca2 = df_pca2.dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca2)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BERTIOGA - CENÁRIO 2 (FÍSICA + HISTÓRICO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_2))],
    index=X_cols_2
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BERTIOGA - CENÁRIO 2 (FÍSICA + HISTÓRICO)

Variância no Componente 1 (PC1): 32.56%
Variância no Componente 2 (PC2): 27.29%
Variância Acumulada (PC1 + PC2): 59.85%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.469847   
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.462855   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.443916   
RADIACAO GLOBAL (Kj/m²)                            -0.335032   
VENTO, RAJADA MAXIMA (m/s)                         -0.249648   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.224700   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.222188   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.218769   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.207318   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.051027   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.008693   



## Teste com variáveis selecionadas + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 3 (Física Limpa + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca3 = df[X_cols_3].copy()
df_pca3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca3['RADIACAO GLOBAL (Kj/m²)'] = df_pca3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca3 = df_pca3.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_pca3[col_v] = df_pca3[col_v].fillna(df_pca3[col_v].median())

df_pca3 = df_pca3.dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca3)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BERTIOGA - CENÁRIO 3 (FÍSICA LIMPA + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para o PC1
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_3))],
    index=X_cols_3
)

loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BERTIOGA - CENÁRIO 3 (FÍSICA LIMPA + TEMPO)

Variância no Componente 1 (PC1): 26.17%
Variância no Componente 2 (PC2): 21.15%
Variância Acumulada (PC1 + PC2): 47.33%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
RADIACAO GLOBAL (Kj/m²)                             0.495166   
UMIDADE RELATIVA DO AR, HORARIA (%)                -0.468406   
VENTO, VELOCIDADE HORARIA (m/s)                     0.423608   
Hora                                                0.419582   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.366827   
Mes                                                -0.209855   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI... -0.031607   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.017717   

                                                    Impacto_Absoluto_PC1  
RADIACAO GLOBAL (Kj/m²)                                         0.495166  
UMIDADE RELATIVA DO AR, HORARIA (%)     

## Teste com todas as variáveis (Sem filtro) + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 4 (Todas as 13 Variáveis)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca4 = df[X_cols_4].copy()
df_pca4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca4['RADIACAO GLOBAL (Kj/m²)'] = df_pca4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca4 = df_pca4.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_pca4[col_v] = df_pca4[col_v].fillna(df_pca4[col_v].median())

df_pca4 = df_pca4.dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca4)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BERTIOGA - CENÁRIO 4 (CAOS TOTAL)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para o PC1
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_4))],
    index=X_cols_4
)

loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BERTIOGA - CENÁRIO 4 (CAOS TOTAL)

Variância no Componente 1 (PC1): 29.22%
Variância no Componente 2 (PC2): 23.25%
Variância Acumulada (PC1 + PC2): 52.47%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.464310   
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.462990   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.443768   
RADIACAO GLOBAL (Kj/m²)                            -0.338498   
Hora                                               -0.269179   
VENTO, RAJADA MAXIMA (m/s)                         -0.234774   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.199761   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.171750   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.169415   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.166117   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.063047   
Mes      

## Teste com variáveis selecionadas - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 1 (Física pura, sem variáveis de tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento do Cinturão de Segurança (Padrão Litoral)
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both')

colunas_vento = ['VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, VELOCIDADE HORARIA (m/s)']
for col_v in colunas_vento:
    df_model[col_v] = df_model[col_v].fillna(df_model[col_v].median())

df_model = df_model.dropna(subset=[target_col])

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com os Hiperparâmetros Otimizados de Bertioga (Cenário 1)
modelo_tunado = XGBRegressor(
    subsample=0.7,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.2,
    colsample_bytree=1.0,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 1)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 1)")
print("Foco: Capacidade Real de Previsão Sem Espiar o Futuro")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 1)
Foco: Estabilidade Estrutural das Regras Físicas
Dobra Aleatória 1: R² = 0.7716
Dobra Aleatória 2: R² = 0.7640
Dobra Aleatória 3: R² = 0.7668
Dobra Aleatória 4: R² = 0.7781
Dobra Aleatória 5: R² = 0.7591

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7679
Desvio Padrão das Dobras: 0.0065

MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 1)
Foco: Capacidade Real de Previsão Sem Espiar o Futuro
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = -0.1806
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.0055
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.1571
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.5400
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6323

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.2287
Desvio Padrão Temporal: 0.3122


## Teste com todas as variáveis (Sem filtro) - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 2 (11 Variáveis Físicas e Históricas)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df[[target_col] + X_cols_2].copy()

# Tratamento do Cinturão de Segurança Litorâneo
df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model_2[col_v] = df_model_2[col_v].fillna(df_model_2[col_v].median())

df_model_2 = df_model_2.dropna(subset=[target_col])

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros do C2 (Árvore Rasa Conservadora)
modelo_tunado_c2 = XGBRegressor(
    subsample=0.7,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.2,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 2)")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_c2, X_2, y_2, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 2)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_2)):
    X_train, X_test = X_2.iloc[train_index], X_2.iloc[test_index]
    y_train, y_test = y_2.iloc[train_index], y_2.iloc[test_index]

    modelo_tunado_c2.fit(X_train, y_train)
    score = modelo_tunado_c2.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 2)
Dobra Aleatória 1: R² = 0.7779
Dobra Aleatória 2: R² = 0.7770
Dobra Aleatória 3: R² = 0.7764
Dobra Aleatória 4: R² = 0.7813
Dobra Aleatória 5: R² = 0.7670

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7759
Desvio Padrão: 0.0048

MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 2)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = -0.1227
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.0313
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.1331
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.5566
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6126

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.2422
Desvio Padrão Temporal: 0.2917


## Teste com variáveis selecionadas + Hora e mês - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 3 (Física Limpa + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df[[target_col] + X_cols_3].copy()

# Tratamento do Cinturão de Segurança (Padrão Litoral)
df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]
for col_v in colunas_vento:
    df_model_3[col_v] = df_model_3[col_v].fillna(df_model_3[col_v].median())

df_model_3 = df_model_3.dropna(subset=[target_col])

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros do C3 (Árvore Rasa)
modelo_tunado_c3 = XGBRegressor(
    subsample=0.7,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.2,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 3)")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_c3, X_3, y_3, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 3)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_3)):
    X_train, X_test = X_3.iloc[train_index], X_3.iloc[test_index]
    y_train, y_test = y_3.iloc[train_index], y_3.iloc[test_index]

    modelo_tunado_c3.fit(X_train, y_train)
    score = modelo_tunado_c3.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 3)
Dobra Aleatória 1: R² = 0.8968
Dobra Aleatória 2: R² = 0.8931
Dobra Aleatória 3: R² = 0.8850
Dobra Aleatória 4: R² = 0.8918
Dobra Aleatória 5: R² = 0.8841

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8902
Desvio Padrão: 0.0049

MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 3)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = -0.2870
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.4923
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.5595
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7062
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6350

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.4212
Desvio Padrão Temporal: 0.3613


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Bertioga
nome_arquivo = 'Bertioga_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 4 (Caos Total: 13 Variáveis)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df[[target_col] + X_cols_4].copy()

# Tratamento do Cinturão de Segurança Litorâneo
df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear', limit_direction='both')

colunas_vento = [
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]
for col_v in colunas_vento:
    df_model_4[col_v] = df_model_4[col_v].fillna(df_model_4[col_v].median())

df_model_4 = df_model_4.dropna(subset=[target_col])

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros do C4
modelo_tunado_c4 = XGBRegressor(
    subsample=0.9,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    colsample_bytree=0.9,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 4)")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_c4, X_4, y_4, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 4)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_4)):
    X_train, X_test = X_4.iloc[train_index], X_4.iloc[test_index]
    y_train, y_test = y_4.iloc[train_index], y_4.iloc[test_index]

    modelo_tunado_c4.fit(X_train, y_train)
    score = modelo_tunado_c4.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BERTIOGA (CENÁRIO 4)
Dobra Aleatória 1: R² = 0.8912
Dobra Aleatória 2: R² = 0.8865
Dobra Aleatória 3: R² = 0.8820
Dobra Aleatória 4: R² = 0.8867
Dobra Aleatória 5: R² = 0.8783

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8850
Desvio Padrão: 0.0044

MÉTODO B: TIME SERIES SPLIT - BERTIOGA (CENÁRIO 4)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = -0.3468
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.5378
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.5818
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.6739
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6529

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.4199
Desvio Padrão Temporal: 0.3865
